# 02 — Preparing the Occupancy Modelling Data

## Part 1: Load and check the selected building space

**Current checkpoint:** this notebook loads the raw measurements for the selected space, restricts them to the agreed study period and displays basic data-quality checks. It does **not** yet create forecasting examples, split examples, train models or evaluate forecast accuracy.

The selected task is to forecast sensor-reported headcount 30 minutes ahead. Forecast opportunities will be scheduled every 15 minutes. The proposed information for each forecast is the current headcount, headcounts 15 and 30 minutes earlier, time of day and day of week. The five-minute measurement-age allowance remains provisional.

| Partition | Agreed local calendar period |
|---|---|
| Training | March 2024–August 2025 |
| Validation | September–December 2025 |
| Final testing | January–March 2026 |

Later sections will construct these examples, check that their targets respect the split boundaries and save the prepared partitions. Reading raw test-period records for predetermined preparation checks is not model fitting or model selection. No test accuracy is calculated here.

Rewrite the explanations in your own words while preserving the meaning. Do not use an occupied/unoccupied status as a substitute for a missing headcount.

## 1. Import the tools

`pandas` handles tables and timestamps, `zipfile` opens the supplied ZIP, and `Path` represents file locations. `display` presents small tables. These are the same tools already used in the feasibility notebook; no additional packages are required for this section.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
from IPython.display import display

print("pandas version:", pd.__version__)

pandas version: 3.0.6


## 2. Identify the data file and building space

This notebook should be saved inside `notebooks/`. The relative path `../data/DataSet2.zip` moves up one folder and then enters `data/`, matching the working-directory arrangement used in the feasibility notebook.

`227f1068` was an abbreviated display label. Filtering uses the **full identifier**, so this code selects the intended monitored space exactly. The location in the ZIP is the same as before.

Check that **ZIP found** is `True` before continuing. These commands do not modify or extract the original ZIP.

In [2]:
zip_path = Path("../data/DataSet2.zip")

csv_name = (
    "DataSet2_Env_and_Occupancy/"
    "5occupancySensor_MayToDec2024_9MRows/"
    "5occupancySensor_MayToDec2024_9MRows.csv"
)

space_id = "227f1068-9e41-469f-bef0-275f27a0bdba"

print("Working folder:", Path.cwd())
print("ZIP found:", zip_path.exists())
print("Selected space:", space_id)

Working folder: c:\Users\chris\OneDrive\Desktop\MMA3001\Project\mma3001_project\notebooks
ZIP found: True
Selected space: 227f1068-9e41-469f-bef0-275f27a0bdba


## 3. Record the agreed date boundaries

These boundaries use Melbourne local time. Each period includes its start but excludes its end. For example, the overall study includes records before midnight at the start of 1 April 2026, so it includes all of 31 March but none of April.

`validation_start` and `test_start` record the agreed boundaries for later splitting. Defining them here does not yet split the raw data or create prediction examples. Likewise, defining `horizon` does not make a prediction.

In [3]:
train_start = pd.Timestamp("2024-03-01", tz="Australia/Melbourne")
validation_start = pd.Timestamp("2025-09-01", tz="Australia/Melbourne")
test_start = pd.Timestamp("2026-01-01", tz="Australia/Melbourne")
study_end = pd.Timestamp("2026-04-01", tz="Australia/Melbourne")

horizon = pd.Timedelta(minutes=30)
max_age = pd.Timedelta(minutes=5)
forecast_frequency = "15min"

print("Study starts:", train_start)
print("Validation starts:", validation_start)
print("Testing starts:", test_start)
print("Study ends (excluded):", study_end)

Study starts: 2024-03-01 00:00:00+11:00
Validation starts: 2025-09-01 00:00:00+10:00
Testing starts: 2026-01-01 00:00:00+11:00
Study ends (excluded): 2026-04-01 00:00:00+11:00


## 4. Read the selected space, one chunk at a time

This reuses the feasibility notebook's approach. Each chunk contains at most 200,000 raw rows. This time, the filter keeps rows whose `floorspaceid` equals the selected ID, rather than filtering for October.

We still read through the complete CSV, because records for a space can occur throughout the file. Only the selected rows are retained in `space_parts`, then combined into `raw_space`. We keep `occupancystatus` for checking records; it is not an agreed model input.

The original data is unchanged. The number printed is a **raw record count**, not a count of usable forecasting examples.

In [4]:
columns = ["floorspaceid", "collecteddate", "headcount", "occupancystatus"]
space_parts = []

with zipfile.ZipFile(zip_path) as zip_file:
    with zip_file.open(csv_name) as csv_file:
        for chunk in pd.read_csv(csv_file, usecols=columns, chunksize=200000):
            selected_rows = chunk[chunk["floorspaceid"] == space_id]
            space_parts.append(selected_rows)

raw_space = pd.concat(space_parts, ignore_index=True)

print("Raw records for the selected space:", len(raw_space))
print("Number of space IDs:", raw_space["floorspaceid"].nunique())

Raw records for the selected space: 1500394
Number of space IDs: 1


## 5. Convert dates and headcounts, then check them

Text dates are converted to timestamps before comparison. `format="ISO8601"` matches the supplied timestamp notation; `utc=True` puts offsets on a consistent time basis. The additional `local_time` column expresses the same instants in Melbourne local time.

`errors="coerce"` marks values that cannot be converted as missing. We count them and stop rather than silently discard them. Each `assert` is a check: if its condition is false, Python stops with the explanatory message. This makes a changed or unexpected dataset visible before later preparation.

The non-negative, integer check is appropriate because these are raw reported people counts, not regression predictions. `% 1` is the remainder after division by one; it is zero for whole numbers. It also flags non-finite numeric values rather than allowing them into the data unnoticed.

These checks establish internal format consistency, not whether the sensor physically counted people correctly.

In [5]:
raw_space["timestamp"] = pd.to_datetime(
    raw_space["collecteddate"],
    format="ISO8601",
    utc=True,
    errors="coerce"
)

raw_space["local_time"] = raw_space["timestamp"].dt.tz_convert(
    "Australia/Melbourne"
)

raw_space["headcount"] = pd.to_numeric(
    raw_space["headcount"], errors="coerce"
)

invalid_times = raw_space["timestamp"].isna().sum()
invalid_counts = raw_space["headcount"].isna().sum()
negative_counts = (raw_space["headcount"] < 0).sum()
non_integer_counts = (raw_space["headcount"].dropna() % 1 != 0).sum()

print("Invalid timestamps:", invalid_times)
print("Missing or non-numeric headcounts:", invalid_counts)
print("Negative headcounts:", negative_counts)
print("Non-integer or non-finite headcounts:", non_integer_counts)

assert len(raw_space) > 0, "No rows found. Check the space ID."
assert invalid_times == 0, "Stop: inspect invalid timestamps before continuing."
assert invalid_counts == 0, "Stop: inspect invalid headcounts before continuing."
assert negative_counts == 0, "Stop: inspect negative headcounts before continuing."
assert non_integer_counts == 0, "Stop: inspect non-integer headcounts before continuing."

Invalid timestamps: 0
Missing or non-numeric headcounts: 0
Negative headcounts: 0
Non-integer or non-finite headcounts: 0


## 6. Keep the study period and inspect repeated timestamps

The two date comparisons keep records at or after the study start and strictly before the study end. `&` means both conditions must hold. We then sort chronologically and reset the ordinary row numbers; neither operation adds observations.

This is an overall date filter, **not yet** a train/validation/test split. Raw records are not the same as forecasting examples.

A repeated timestamp could contain agreeing headcounts or conflicting ones. We display every member of each repeated-timestamp group using `keep=False`, so no disagreement is hidden. No repeated record is removed in this cell.

For the supplied ZIP, the two repeated timestamps each have two records with headcount zero, but different occupancy-status labels. They are not identical full records. A later preparation step should explicitly decide how to represent these agreeing headcount observations before timestamp-based lookups.

In [6]:
space_data = raw_space[
    (raw_space["local_time"] >= train_start)
    & (raw_space["local_time"] < study_end)
].copy()

space_data = space_data.sort_values("timestamp")
space_data = space_data.reset_index(drop=True)

print("Records in the study period:", len(space_data))
print("First observation:", space_data["local_time"].min())
print("Last observation:", space_data["local_time"].max())
print("Extra rows at repeated timestamps:", space_data["timestamp"].duplicated().sum())

print("First five observations:")
display(space_data[["local_time", "headcount"]].head())

repeated_times = space_data[
    space_data["timestamp"].duplicated(keep=False)
]

print("All records involved in repeated timestamps:")
display(repeated_times[["local_time", "headcount", "occupancystatus"]])

Records in the study period: 1483464
First observation: 2024-03-03 15:06:30+11:00
Last observation: 2026-03-31 23:17:52+11:00
Extra rows at repeated timestamps: 2
First five observations:


,local_time,headcount
0,2024-03-03 15:06:30+11:00,0
1,2024-03-03 15:08:18+11:00,1
2,2024-03-03 15:08:50+11:00,0
3,2024-03-03 15:09:02+11:00,1
4,2024-03-03 15:09:42+11:00,0


All records involved in repeated timestamps:


,local_time,headcount,occupancystatus
56782,2024-04-05 09:13:42+11:00,0,RecentlyOccupied
56783,2024-04-05 09:13:42+11:00,0,NotOccupied
208917,2024-05-23 09:19:28+10:00,0,RecentlyOccupied
208918,2024-05-23 09:19:28+10:00,0,NotOccupied


## Checkpoint: what has and has not been established

For the supplied ZIP, this section is expected to find 1,500,394 raw records for the selected space, and 1,483,464 records within the agreed study period. The first retained observation is 3 March 2024 at 15:06:30 (+11:00); there are no observations for this space on 1–2 March. We have not filled those days or interpreted them as zero occupancy.

The final retained observation is 31 March 2026 at 23:17:52 (+11:00). All four format/validity counts should be zero. Two extra records share timestamps with other records, producing four displayed rows.

Matching these outputs checks that the extraction has reproduced the same subset. It does not establish forecasting accuracy or confirm physical room identities.

**Next sections, not yet implemented:** resolve the repeated headcount observations, create the 15-minute forecast schedule, obtain the current and lagged inputs plus the 30-minute target using the stated measurement-age rule, count missing examples, apply the chronological split with target-boundary checks, and save the prepared data locally.

Feature scaling and model fitting belong in the model notebooks. Scaling must be learned from the training partition only, rather than from all data before splitting.

## Reference documentation

The code follows the existing project notebook and the agreed project choices. API references:

- [pandas CSV reading](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [pandas timestamp parsing](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas timezone conversion](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.tz_convert.html)
- [pandas repeated-row checks](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)
- [scikit-learn guidance on preprocessing leakage](https://scikit-learn.org/stable/common_pitfalls.html)

Draft code and explanations were generated with AI assistance. Running the notebook and interpreting its checks remain part of the student's verification; this note does not claim that the student has already completed those checks.